In [1]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

from src.classifier.data import CATEGORIES, clean_text
from src.classifier.split import load_split, apply_split

df = pd.read_csv("data/opp115_segments.csv")
df["text"] = df["text"].map(clean_text)
parts = apply_split(df, load_split())
yva = parts["val"][CATEGORIES].values
yte = parts["test"][CATEGORIES].values

# baseline probabilities
m = joblib.load("artifacts/classifier/baseline.joblib")
base_val = m["clf"].predict_proba(m["vec"].transform(parts["val"]["text"]))
base_test = m["clf"].predict_proba(m["vec"].transform(parts["test"]["text"]))

# BERT probabilities (from Colab)
bert_val = np.load("results/val_probs_bert_v2.npy")
bert_test = np.load("results/test_probs_bert_v2.npy")

In [2]:
grid = np.arange(0.05, 0.95, 0.05)


def tune(probs, y):
    out = []
    for j in range(y.shape[1]):
        s = [f1_score(y[:, j], (probs[:, j] > t).astype(int), zero_division=0) for t in grid]
        out.append(float(grid[int(np.argmax(s))]))
    return out


def f1s(probs, y, thr):
    return [f1_score(y[:, j], (probs[:, j] > thr[j]).astype(int), zero_division=0)
            for j in range(y.shape[1])]


half = [0.5] * len(CATEGORIES)
tab = pd.DataFrame({
    "baseline @0.5": f1s(base_test, yte, half),
    "baseline tuned": f1s(base_test, yte, tune(base_val, yva)),
    "BERT @0.5": f1s(bert_test, yte, half),
    "BERT tuned": f1s(bert_test, yte, tune(bert_val, yva)),
}, index=CATEGORIES)
tab["test support"] = yte.sum(axis=0)
tab.loc["macro-F1"] = tab.drop(columns="test support").mean()
tab = tab.round(3)
tab.to_csv("results/e4_test_comparison.csv")
tab

,baseline @0.5,baseline tuned,BERT @0.5,BERT tuned,test support
First Party Collection/Use,0.810,0.817,0.865,0.881,196.0
Third Party Sharing/Collection,0.736,0.766,0.833,0.833,144.0
User Choice/Control,0.693,0.685,0.685,0.676,62.0
"User Access, Edit and Deletion",0.741,0.632,0.808,0.833,25.0
Data Retention,0.462,0.462,0.588,0.588,10.0
Data Security,0.795,0.787,0.886,0.865,37.0
Policy Change,0.889,0.884,0.776,0.809,23.0
Do Not Track,0.800,0.667,0.800,0.800,6.0
International and Specific Audiences,0.832,0.825,0.869,0.841,61.0
Other,0.667,0.622,0.718,0.700,212.0


In [3]:
def macro(probs, y, thr=0.5):
    return np.mean([f1_score(y[:, j], (probs[:, j] > thr).astype(int), zero_division=0)
                    for j in range(y.shape[1])])

print("baseline val:", round(macro(base_val, yva), 3))
print("BERT val:    ", round(macro(bert_val, yva), 3))
for w in [0.5, 0.6, 0.7, 0.8, 0.9]:
    print("BERT weight", w, round(macro(w * bert_val + (1 - w) * base_val, yva), 3))

baseline val: 0.763
BERT val:     0.837
BERT weight 0.5 0.837
BERT weight 0.6 0.833
BERT weight 0.7 0.832
BERT weight 0.8 0.84
BERT weight 0.9 0.837
